In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
PMADS_data = pd.read_excel("../PMADS.xlsx")

In [4]:
Filtered_data_Melanoma_ARi_Phospho = PMADS_data[(PMADS_data.loc[:,"cell line"]== "WM793") & (PMADS_data.loc[:,"PTM"]== "phosphorylation") 
           & (PMADS_data.loc[:,"Disease"]== "Melanoma") & (PMADS_data.loc[:,"species"]== "human") & ((PMADS_data.loc[:,"Drug"]== "ARi + FUT4 OE") | ((PMADS_data.loc[:,"Drug"]== "ARi") & (PMADS_data.loc[:,"Disease_fine"]== "FUT4 OE")))]


In [5]:
Filtered_data_Melanoma_ARi_Phospho 

,ID,Version,PMID_sentenceIndex,PMID,Sentence,Protein_uniport,Protein,protein_entry,PTM,gene_manual,...,Note,Confidence score,Confidence level,Drug_class,Disease_category,Regulatory Class,Pattern,PTM-regulation,Disease-regulation,Status
43600,acc38826,202507,PRIDE title,PXD047335,ANDROGEN DRIVES MELANOMA INVASIVENESS AND META...,A0FGR8,Extended synaptotagmin-2,ESYT2_HUMAN,phosphorylation,ESYT2,...,1366_OE_ARi - ctl,3.0,Low,Combination therapy,Cancer,Impairing,3,ptm-down,disease-good,Inferred
43601,acc38827,202507,PRIDE title,PXD047335,ANDROGEN DRIVES MELANOMA INVASIVENESS AND META...,A0FGR8,Extended synaptotagmin-2,ESYT2_HUMAN,phosphorylation,ESYT2,...,1366_OE_ARi - ctl,3.0,Low,Combination therapy,Cancer,Impairing,3,ptm-down,disease-good,Inferred
43602,acc38828,202507,PRIDE title,PXD047335,ANDROGEN DRIVES MELANOMA INVASIVENESS AND META...,A0FGR8,Extended synaptotagmin-2,ESYT2_HUMAN,phosphorylation,ESYT2,...,1366_OE_ARi - ctl,3.0,Low,Combination therapy,Cancer,Impairing,3,ptm-down,disease-good,Inferred
43603,acc38829,202507,PRIDE title,PXD047335,ANDROGEN DRIVES MELANOMA INVASIVENESS AND META...,A0JNW5,Bridge-like lipid transfer protein family memb...,BLT3B_HUMAN,phosphorylation,BLTP3B,...,1366_OE_ARi - ctl,3.0,Low,Combination therapy,Cancer,Enhancing,3,ptm-up,disease-good,Inferred
43604,acc38830,202507,PRIDE title,PXD047335,ANDROGEN DRIVES MELANOMA INVASIVENESS AND META...,A1L170,Uncharacterized protein C1orf226,CA226_HUMAN,phosphorylation,C1orf226,...,1366_OE_ARi - ctl,3.0,Low,Combination therapy,Cancer,Impairing,3,ptm-down,disease-good,Inferred
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46523,acc41749,202507,PRIDE title,PXD047335,ANDROGEN DRIVES MELANOMA INVASIVENESS AND META...,P49840,Glycogen synthase kinase-3 alpha,GSK3A_HUMAN,phosphorylation,GSK3A,...,ARi - ctl,3.0,Low,Unclassified,Cancer,Impairing,3,ptm-down,disease-good,Inferred
46524,acc41750,202507,PRIDE title,PXD047335,ANDROGEN DRIVES MELANOMA INVASIVENESS AND META...,P49841,Glycogen synthase kinase-3 beta,GSK3B_HUMAN,phosphorylation,GSK3B,...,ARi - ctl,3.0,Low,Unclassified,Cancer,Impairing,3,ptm-down,disease-good,Inferred
46525,acc41751,202507,PRIDE title,PXD047335,ANDROGEN DRIVES MELANOMA INVASIVENESS AND META...,P62995,Transformer-2 protein homolog beta,TRA2B_HUMAN,phosphorylation,TRA2B,...,ARi - ctl,3.0,Low,Unclassified,Cancer,Impairing,3,ptm-down,disease-good,Inferred
46526,acc41752,202507,PRIDE title,PXD047335,ANDROGEN DRIVES MELANOMA INVASIVENESS AND META...,Q13595,Transformer-2 protein homolog alpha,TRA2A_HUMAN,phosphorylation,TRA2A,...,ARi - ctl,3.0,Low,Unclassified,Cancer,Impairing,3,ptm-down,disease-good,Inferred


In [6]:
uniref_annot = pd.read_csv("/home/saishyam/Protein_dynamics/all_uniref/uniprotkb_AND_reviewed_true_AND_model_o_2026_01_16.tsv", sep="\t", index_col=0)

In [7]:
# Create mapping from Entry Name -> Sequence
seq_map = uniref_annot.set_index("Entry Name")["Sequence"]

# Add Sequence column to Filtered_data_Prostate_Palbocilib_Phospho
Filtered_data_Melanoma_ARi_Phospho ["Sequence"] = (
    Filtered_data_Melanoma_ARi_Phospho["protein_entry"]
    .map(seq_map)
)

/tmp/ipykernel_2057568/3673790739.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Filtered_data_Melanoma_ARi_Phospho ["Sequence"] = (


In [8]:
Filtered_data_Melanoma_ARi_Phospho.loc[:,"Sequence"].isna().sum()

np.int64(20)

In [9]:
Filtered_data_Melanoma_ARi_Phospho = (
    Filtered_data_Melanoma_ARi_Phospho 
    .dropna(subset=["Sequence"])
)

In [10]:
Filtered_data_Melanoma_ARi_Phospho .columns

Index(['ID', 'Version', 'PMID_sentenceIndex', 'PMID', 'Sentence',
       'Protein_uniport', 'Protein', 'protein_entry', 'PTM', 'gene_manual',
       'Site', 'species', 'Disease', 'Disease_fine', 'cell line', 'Drug_info',
       'Drug', 'KW_info', 'KW', 'Pass', 'Note', 'Confidence score',
       'Confidence level', 'Drug_class', 'Disease_category',
       'Regulatory Class', 'Pattern', 'PTM-regulation', 'Disease-regulation',
       'Status', 'Sequence'],
      dtype='object')

In [11]:
import pandas as pd
import re

WINDOW_SIZE = 15
HALF_WINDOW = WINDOW_SIZE // 2  # 7 residues each side

# Mapping from site prefix -> amino acid letter
SITE_MAP = {
    "Ser": "S",
    "Thr": "T",
    "Tyr": "Y"
}

def extract_window_and_verify(sequence, site):
    """
    Extract 15-mer centered window and verify
    the phosphosite residue matches sequence.
    
    Returns:
        window, residue_match, actual_residue
    """

    if pd.isna(sequence) or pd.isna(site):
        return None, False, None

    # Extract residue type + numeric position
    match = re.match(r'([A-Za-z]+)(\d+)', str(site))
    if not match:
        return None, False, None

    residue_name = match.group(1)
    pos = int(match.group(2))

    expected_residue = SITE_MAP.get(residue_name)

    if expected_residue is None:
        return None, False, None

    # Convert to 0-based index
    idx = pos - 1

    # Bounds check
    if idx < 0 or idx >= len(sequence):
        return None, False, None

    actual_residue = sequence[idx]

    # Verify residue identity
    residue_match = (actual_residue == expected_residue)

    # Extract window
    start = max(0, idx - HALF_WINDOW)
    end = min(len(sequence), idx + HALF_WINDOW + 1)

    window = sequence[start:end]

    # Pad termini
    left_pad = "_" * (HALF_WINDOW - (idx - start))
    right_pad = "_" * (HALF_WINDOW - (end - idx - 1))

    window = left_pad + window + right_pad

    return window, residue_match, actual_residue


# Apply to dataframe
results = Filtered_data_Melanoma_ARi_Phospho .apply(
    lambda row: extract_window_and_verify(
        row["Sequence"],
        row["Site"]
    ),
    axis=1
)

# Add columns
Filtered_data_Melanoma_ARi_Phospho[
    ["Site_15mer", "Residue_Match", "Actual_Residue"]
] = pd.DataFrame(results.tolist(),
                 index=Filtered_data_Melanoma_ARi_Phospho.index)

# View problematic rows
mismatch_df = Filtered_data_Melanoma_ARi_Phospho[
    ~Filtered_data_Melanoma_ARi_Phospho["Residue_Match"]
]

print("Number of mismatches:", mismatch_df.shape[0])

mismatch_df[
    ["Site", "Actual_Residue", "Sequence"]
].head()

Number of mismatches: 0


/tmp/ipykernel_2057568/3112019209.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Filtered_data_Melanoma_ARi_Phospho[
/tmp/ipykernel_2057568/3112019209.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Filtered_data_Melanoma_ARi_Phospho[
/tmp/ipykernel_2057568/3112019209.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

,Site,Actual_Residue,Sequence


In [12]:
Filtered_data_Melanoma_ARi_Phospho[Filtered_data_Melanoma_ARi_Phospho.loc[:,"Residue_Match"] == False][
    ["Site", "Actual_Residue", "Sequence"]
]

,Site,Actual_Residue,Sequence


In [13]:
# Keep only rows where residue verification passed
Filtered_data_Melanoma_ARi_Phospho = (
    Filtered_data_Melanoma_ARi_Phospho[
        Filtered_data_Melanoma_ARi_Phospho["Residue_Match"] == True
    ]
    .copy()
)

# # Optional: reset index
# Filtered_data_Prostate_Palbocilib_Phospho.reset_index(drop=True, inplace=True)

In [14]:
Filtered_data_Melanoma_ARi_Phospho.to_csv("Filtered_data_Melanoma_ARi_Phospho_WM793.csv", index=False)

In [64]:
pwd

'/home/saishyam/Protein_dynamics/Dynamic_properties/PDMAS_drug_response/prostate_phosphorylation_lncap_palbociclib_analysis'

In [ ]:
'/home/saishyam/Protein_dynamics/Dynamic_properties/PDMAS_drug_response/prostate_phosphorylation_lncap_palbociclib_analysis/Filtered_data_Prostate_Palbocilib_Phospho.csv'

In [65]:
Filtered_data_Prostate_Palbocilib_Phospho

,ID,Version,PMID_sentenceIndex,PMID,Sentence,Protein_uniport,Protein,protein_entry,PTM,gene_manual,...,Disease_category,Regulatory Class,Pattern,PTM-regulation,Disease-regulation,Status,Sequence,Site_15mer,Residue_Match,Actual_Residue
28510,acc23736,202507,PRIDE title,PXD006561,CDK4/6 inhibitor resistance in prostate cancer,Q12774,Rho guanine nucleotide exchange factor 5,ARHG5_HUMAN,phosphorylation,ARHGEF5,...,Cancer,Enhancing,3,ptm-down,disease-bad,Inferred,MEAEEAQRGASPPISAIEEFSIIPEAPMRSSQVSALGLEAQEDEDP...,ESRAEELSPAALSPS,True,S
28513,acc23739,202507,PRIDE title,PXD006561,CDK4/6 inhibitor resistance in prostate cancer,Q12774,Rho guanine nucleotide exchange factor 5,ARHG5_HUMAN,phosphorylation,ARHGEF5,...,Cancer,Enhancing,3,ptm-down,disease-bad,Inferred,MEAEEAQRGASPPISAIEEFSIIPEAPMRSSQVSALGLEAQEDEDP...,ELSPAALSPSLEPIR,True,S
28517,acc23743,202507,PRIDE title,PXD006561,CDK4/6 inhibitor resistance in prostate cancer,A0AAG2TJA1,Uncharacterized protein,YM012_HUMAN,phosphorylation,A0AAG2TJA1,...,Cancer,Enhancing,3,ptm-down,disease-bad,Inferred,MRRPSTASLTRTPSRASPTRMPSRASLKMTPFRASLTKMESTALLR...,SSLTRTPSMASLTRT,True,S
28519,acc23745,202507,PRIDE title,PXD006561,CDK4/6 inhibitor resistance in prostate cancer,A0AAG2TJA1,Uncharacterized protein,YM012_HUMAN,phosphorylation,A0AAG2TJA1,...,Cancer,Enhancing,3,ptm-down,disease-bad,Inferred,MRRPSTASLTRTPSRASPTRMPSRASLKMTPFRASLTKMESTALLR...,ASLTRTPSMVSLKRS,True,S
28520,acc23746,202507,PRIDE title,PXD006561,CDK4/6 inhibitor resistance in prostate cancer,A0FGR8,Extended synaptotagmin-2,ESYT2_HUMAN,phosphorylation,ESYT2,...,Cancer,Enhancing,3,ptm-down,disease-bad,Inferred,MTANRDAALSSHRHPGCAQRPRTPTFASSSQRRSAFGFDDGNFPGL...,HDLGRSSSSLLASPG,True,S
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36914,acc32140,202507,PRIDE title,PXD006561,CDK4/6 inhibitor resistance in prostate cancer,Q9NZ63,Splicing factor C9orf78,TLS1_HUMAN,phosphorylation,C9orf78,...,Cancer,Impairing,3,ptm-down,disease-good,Inferred,MPVVRKIFRRRRGDSESEEDEQDSEEVRLKLEETREVQNLRKRPNG...,KNAEDCLYELPENIR,True,Y
36915,acc32141,202507,PRIDE title,PXD006561,CDK4/6 inhibitor resistance in prostate cancer,Q9UDY2,Tight junction protein ZO-2,ZO2_HUMAN,phosphorylation,TJP2,...,Cancer,Enhancing,3,ptm-up,disease-good,Inferred,MPVRGDRGFPPRRELSGWLRAPGMEELIWEQYTVTLQKDSKRGFGI...,AQKHPDIYAVPIKTH,True,Y
36916,acc32142,202507,PRIDE title,PXD006561,CDK4/6 inhibitor resistance in prostate cancer,Q9UQE7,Structural maintenance of chromosomes protein 3,SMC3_HUMAN,phosphorylation,SMC3,...,Cancer,Enhancing,3,ptm-up,disease-good,Inferred,MYIKQVIIQGFRSYRDQTIVDPFSSKHNVIVGRNGSGKSNFFYAIQ...,TAGNRLFYHIVDSDE,True,Y
36917,acc32143,202507,PRIDE title,PXD006561,CDK4/6 inhibitor resistance in prostate cancer,Q9Y237,Peptidyl-prolyl cis-trans isomerase NIMA-inter...,PIN4_HUMAN,phosphorylation,PIN4,...,Cancer,Enhancing,3,ptm-up,disease-good,Inferred,MPPKGKSGSGKAGKGGAASGSDSADKKAQGPKGGGNAVKVRHILCE...,PVKTKFGYHIIMVEG,True,Y


In [72]:
PMADS_data[( PMADS_data.loc[:,"Drug"] == "Palbociclib") & (PMADS_data.loc[:,"cell line"]== "LNCaP") & (PMADS_data.loc[:,"PTM"]== "phosphorylation") 
           & (PMADS_data.loc[:,"Disease"]== "Prostate Cancer") & (PMADS_data.loc[:,"species"]!= "human") ]

,ID,Version,PMID_sentenceIndex,PMID,Sentence,Protein_uniport,Protein,protein_entry,PTM,gene_manual,...,Note,Confidence score,Confidence level,Drug_class,Disease_category,Regulatory Class,Pattern,PTM-regulation,Disease-regulation,Status
30881,acc26107,202507,PRIDE title,PXD006561,CDK4/6 inhibitor resistance in prostate cancer,Q96PC5,A4FU28,MIA2_HUMAN,phosphorylation,CTAGE9,...,Drug treatment resistant,3.0,Low,Kinase inhibitors,Cancer,Impairing,3,ptm-up,disease-bad,Inferred
34684,acc29910,202507,PRIDE title,PXD006561,CDK4/6 inhibitor resistance in prostate cancer,Q96PC5,A4FU28,MIA2_HUMAN,phosphorylation,CTAGE9,...,Drug treatment sensitive,3.0,Low,Kinase inhibitors,Cancer,Enhancing,3,ptm-up,disease-good,Inferred
34691,acc29917,202507,PRIDE title,PXD006561,CDK4/6 inhibitor resistance in prostate cancer,Q96PC5,A4FU28,MIA2_HUMAN,phosphorylation,CTAGE9,...,Drug treatment sensitive,3.0,Low,Kinase inhibitors,Cancer,Enhancing,3,ptm-up,disease-good,Inferred


In [79]:
PMADS_data[ (PMADS_data.loc[:,"Disease"]== "Breast Cancer") & (PMADS_data.loc[:,"species"]== "human") & (PMADS_data.loc[:,"cell line"]== "MCF7") & (PMADS_data.loc[:,"PTM"]== "phosphorylation") ].loc[:,"PTM-regulation"].value_counts()

PTM-regulation
ptm-up      1140
ptm-down     437
Name: count, dtype: int64

In [75]:
PMADS_data[ (PMADS_data.loc[:,"species"]== "human") ].loc[:,"Disease"].value_counts()

Disease
Prostate Cancer                         13481
Breast Cancer                           12865
Melanoma                                 6653
Lung Cancer                              6089
Myeloma                                  1452
                                        ...  
IPF                                         1
Endothelial Cell Injury, Apoptosis          1
Dysesthesia                                 1
Ovarian Endometriosis                       1
Cerebral Ischemia-Reperfusion Injury        1
Name: count, Length: 118, dtype: int64